In [1]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='file:///c:/Users/MSI/Desktop/MLOps/MLOps_ZoomCamp_Course/02_Experiment_Tracking/mlruns/1', creation_time=1794146536551, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794146536551, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [2]:
import pandas as pd

df_train = pd.read_parquet("../Data/green_tripdata_2021-01.parquet")

df_test = pd.read_parquet("../Data/green_tripdata_2021-02.parquet")

In [3]:
df_train["duration"] = (
  df_train["lpep_dropoff_datetime"] - df_train["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df_test["duration"] = (
  df_test["lpep_dropoff_datetime"] - df_test["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df_train = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)]
df_test = df_test[(df_test.duration >= 1) & (df_test.duration <= 60)] 

In [4]:
from sklearn.feature_extraction import DictVectorizer

categorical = ["PULocationID", "DOLocationID"]
numerical = ["trip_distance"]

train_dicts = df_train[categorical + numerical].to_dict(orient="records")
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

test_dicts = df_test[categorical + numerical].to_dict(orient="records")
X_test = dv.transform(test_dicts)

y_train = df_train["duration"].values
y_test = df_test["duration"].values

In [5]:
import numpy as np

for X in (X_train, X_test):
  X.indices = X.indices.astype(np.int32)
  X.indptr = X.indptr.astype(np.int32)

In [6]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)

print("train", root_mean_squared_error(y_train, model.predict(X_train)))
print("val  ", root_mean_squared_error(y_test, model.predict(X_test)))

train 11.449081717462942
val   12.078673198609463


In [7]:
lasso = Lasso(alpha = 0.1).fit(X_train, y_train)
print("lasso val", root_mean_squared_error(y_test, lasso.predict(X_test)))

lasso val 12.07870203052328


In [8]:
with mlflow.start_run(run_name="linear_regression_baseline"):
  mlflow.log_param("model", "linear_regression")
  mlflow.log_param("train_data", "yellow_tripdata_2021-01.parquet")
  mlflow.log_param("valid_data", "yellow_tripdata_2021-02.parquet")
  mlflow.log_metric("rmse", root_mean_squared_error(y_test, model.predict(X_test)))
  mlflow.log_metric("mae", mean_absolute_error(y_test, model.predict(X_test)))
  mlflow.log_metric("r2", r2_score(y_test, model.predict(X_test)))
  mlflow.log_metric("train_rmse", root_mean_squared_error(y_train, model.predict(X_train)))

In [9]:
for alpha in [0.001, 0.01, 0.1, 1]:
  with mlflow.start_run(run_name=f"lasso_alpha_{alpha}"):
    mlflow.log_param("model", "lasso")
    mlflow.log_param("alpha", alpha)
    lasso = Lasso(alpha=alpha).fit(X_train, y_train)
    mlflow.log_metric("rmse", root_mean_squared_error(y_test, lasso.predict(X_test)))